TASK 1.1 Read the file the foundations way


In [64]:
import csv
import json
import pandas as pd
import numpy as np
import requests

accepted = []
rejected = []

def parse_row(row):
    row=dict(row)
    row["age"]=int(row["age"])
    row["fee_inr"]=float(row["fee_inr"])
    row["prior_no_shows"]=int(row["prior_no_shows"])
    row["distance_km"]=float(row["distance_km"])

with open("appointments.csv", newline='') as f:
    reader=csv.DictReader(f)
    header=reader.fieldnames
    rows=list(reader)

for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row) 

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

print(f"Accepted rows: {len(accepted)}")
print(f"Rejected rows: {len(rejected)}")

Accepted rows: 988
Rejected rows: 32


TASK 1.2 Inspect and clean, pass 1

In [65]:
df=pd.read_csv("appointments.csv")
print("Shape: ", df.shape, "\n\n")
print("Dataframe Information:", df.info(), "\n\n")
df.isnull().sum()
df.duplicated().sum()
new_df=df.drop_duplicates()                                                                          
new_df.duplicated().sum()
new_df["booking_channel"]=new_df["booking_channel"].str.strip().str.title()
new_df["appt_date"]=pd.to_datetime(new_df["appt_date"], errors="coerce")
new_df["booked_on"]=pd.to_datetime(new_df["booked_on"], errors="coerce")

df2=new_df.dropna(subset=["appt_date", "status"], how="any")
df2.isna().sum()


Shape:  (1020, 14) 


<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 170.6 KB
Dataframe Information: None 




appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           0
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              0
dtype: int64

TASK 1.3 Clean, pass 2: validity, gaps and outliers

In [66]:
df2["age"] = pd.to_numeric(df2["age"], errors="coerce")
df2.loc[(df2["age"] < 1) | (df2["age"] > 100), "age"] = np.nan

df2["age"] = df2["age"].fillna(df2["age"].median()).astype(int)

for col in ["distance_km", "fee_inr"]:
    q1 = df2[col].quantile(0.25)
    q3 = df2[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)

    capped = ((df2[col] < lower) | (df2[col] > upper)).sum()
    print(f"Capped {col} values:", capped)

    df2[col] = np.clip(df2[col], lower, upper)

df2["distance_km"] = (df2["distance_km"].fillna(df2["distance_km"].median()).round(1))
df2["fee_inr"] = df2["fee_inr"].round().astype(int)

Capped distance_km values: 26
Capped fee_inr values: 3


TASK 1.4 Clean, pass 2: validity, gaps and outliers 

In [67]:
df2["lead_days"]=(df2["appt_date"] - df2["booked_on"]).dt.days
df2["no_show"] = df2["status"].apply(lambda x: 1 if x == "No-Show" else 0)

X = df2[["age","distance_km","fee_inr","lead_days"]].to_numpy()


In [68]:
col_mean=X.mean(axis=0)

mask = X[:, 3] > 14

count = mask.sum()
share = mask.mean()

print("Rows with lead_days > 14:", count)
print("Share of rows:", share)

Rows with lead_days > 14: 296
Share of rows: 0.298989898989899


In [69]:
x=df2["distance_km"].to_numpy()
z=(x-np.mean(x))/np.std(x)
outliers=(np.abs(z)>2).sum()
print(outliers)

53


TASK 1.5 Enrich the clinics from a live API

In [70]:
clinics = pd.read_csv("clinics.csv")

clinic_data = []

for _, clinic in clinics.iterrows():
    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }

    temp_c = None
    reason = None

    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10,
            )

            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
                continue

            data = response.json()
            hourly = data.get("hourly")

            if not hourly or not hourly.get("temperature_2m"):
                reason = "hourly temperature data missing"
                continue

            temp_c = hourly["temperature_2m"][0]
            break

        except requests.exceptions.RequestException as e:
            reason = str(e)

    if temp_c is None:
        print(f"{clinic['clinic_name']}: {reason}")

    clinic_data.append(temp_c)

clinics["temp_c"] = clinic_data
clinics.to_csv("clinics_enriched.csv", index=False)

TASK 1.6 Export the clean table and write the SQL load script

In [71]:
df3=df2.sort_values("appointment_id")
df3["booked_on"]=pd.to_datetime(df3["booked_on"]).dt.strftime("%Y-%m-%d")
df3["appt_date"]=pd.to_datetime(df3["appt_date"]).dt.strftime("%Y-%m-%d")

columns=["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender", "distance_km",
         "fee_inr", "booking_channel", "visit_type", "reminder_sent", "prior_no_shows", "status", "lead_days", "no_show"]

df3=df3[columns]
df3.to_csv("appointments_clean.csv", index= False)


In [72]:
print("appointment_id unique:", df3["appointment_id"].is_unique)
print("Missing values per column:")
print(df3.isna().sum())

appointments=pd.read_csv("appointments_clean.csv")
clinics=pd.read_csv("clinics_enriched.csv")

with open("load_data.sql", "w") as f:
    for _, c in clinics.iterrows():

        temp_sql = "NULL" if pd.isna(c["temp_c"]) else str(c["temp_c"])

        f.write(f"INSERT INTO clinics VALUES ("
                f"'{c['clinic_id']}', '{c['clinic_name']}',"
                f"'{c['city']}', '{c['tier']}', '{c['latitude']}', '{c['longitude']}',"
                f"'{temp_sql}');\n")

    for _, a in appointments.iterrows():
        f.write(f"INSERT INTO appointments VALUES("
                f"'{a['appointment_id']}', '{a['patient_id']}', '{a['clinic_id']}', '{a['booked_on']}', '{a['appt_date']}',"
                f"'{a['age']}', '{a['gender']}', '{a['distance_km']}', '{a['fee_inr']}', '{a['booking_channel']}', '{a['visit_type']}',"
                f"'{a['reminder_sent']}', '{a['prior_no_shows']}', '{a['status']}', '{a['lead_days']}', '{a['no_show']}');\n")



appointment_id unique: True
Missing values per column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
